# Table 4 예비 연관성 분석표 생성 노트북

## 1. 목적

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로 중간보고서 4. 예비분석 결과 섹션에 들어갈 예비 연관성 분석표를 생성한다.

목적은 회귀모형 실행 전에 핵심 변수 간 방향성이 가설과 대체로 일치하는지 확인하는 것이다. 모든 분석은 최종 분석용 데이터셋을 읽기만 하며, 원본 데이터 파일은 변경하지 않는다.

사용 방식:
- 노트북 맨 위의 `SAVE_OUTPUTS` 플래그로 CSV/Markdown 저장 여부를 한 번에 관리한다.
- `SAVE_OUTPUTS = True`: 결과를 파일로 저장하고 노트북 내부에도 출력한다.
- `SAVE_OUTPUTS = False`: 파일 저장 없이 노트북 내부에서 결과만 확인한다.


## 2. 데이터 로드 및 변수 확인

최종 분석용 데이터셋은 `working/analysis` 폴더에서만 찾는다. 변수 존재 여부와 결측 N을 먼저 확인한다.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np

# 저장 여부 단일 플래그
# True: CSV/Markdown 파일 저장 + 노트북 내부 출력
# False: 파일 저장 없이 노트북 내부 출력만 수행
SAVE_OUTPUTS = True

try:
    from IPython.display import display, Markdown
except ImportError:
    display = None
    Markdown = None

# 실행 위치가 프로젝트 루트 또는 code 폴더일 수 있으므로 루트를 추정한다.
CWD = Path.cwd()
if (CWD / 'working').exists():
    BASE_DIR = CWD
elif (CWD.parent / 'working').exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError('working 폴더가 있는 프로젝트 루트를 찾지 못했습니다.')

ANALYSIS_DIR = BASE_DIR / 'working' / 'analysis'
OUT_DIR = BASE_DIR / 'output' / 'tables'
if SAVE_OUTPUTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for path in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': path.name,
            'path': path,
            'modified_time': pd.Timestamp(path.stat().st_mtime, unit='s'),
            'size_bytes': path.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('사용 데이터 후보')
display(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']]) if display else print(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']])
print('\n선택된 데이터:', DATA_PATH.relative_to(BASE_DIR))

In [ ]:
# 최종 분석용 데이터셋 읽기
suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
required_vars = [
    'effect_proc_improve', 'effect_average', 'it_org_any', 'ai_use',
    'ai_use_sum', 'it_invest_sum', 'firm_size', 'industry'
]

variable_check = pd.DataFrame([
    {
        '변수명': var,
        '존재 여부': var in df.columns,
        '유효 N': int(df[var].notna().sum()) if var in df.columns else np.nan,
        '결측 N': int(df[var].isna().sum()) if var in df.columns else np.nan,
    }
    for var in required_vars
])
missing_vars = variable_check.loc[~variable_check['존재 여부'], '변수명'].tolist()

print('사용 데이터:', DATA_PATH.relative_to(BASE_DIR))
print('전체 N:', f'{total_n:,}')
display(variable_check) if display else print(variable_check.to_string(index=False))

if missing_vars:
    raise KeyError('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))

## 3. 공통 함수

상관계수 유의수준 표시, 큰 표본 정규근사 p-value, Markdown 표 저장 함수 등을 정의한다. 현재 가상환경에는 `scipy`가 없으므로 p-value는 예비적 참고용 정규근사로 계산한다.

In [ ]:
def norm_two_sided_from_z(z):
    if pd.isna(z):
        return np.nan
    return math.erfc(abs(float(z)) / math.sqrt(2.0))


def p_stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def p_display(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '<.001'
    return f'{p:.3f}'


def corr_p_value(r, n):
    if n <= 3 or pd.isna(r):
        return np.nan
    if abs(r) >= 1:
        return 0.0
    z = math.atanh(float(r)) * math.sqrt(n - 3)
    return norm_two_sided_from_z(z)


def report_corr_cell(r, p):
    if pd.isna(r):
        return ''
    return f'{r:.3f}{p_stars(p)}'


def pearson_r(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def md_table(data: pd.DataFrame) -> str:
    def fmt(value):
        if pd.isna(value):
            return ''
        if isinstance(value, (float, np.floating)):
            return f'{float(value):.3f}'
        return str(value).replace('|', '\\|').replace('\n', '<br>')

    cols = list(data.columns)
    lines = ['| ' + ' | '.join(cols) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in data.iterrows():
        lines.append('| ' + ' | '.join(fmt(row[col]) for col in cols) + ' |')
    return '\n'.join(lines)


def show_table(title, data):
    print(title)
    if display:
        display(data)
    else:
        print(data.to_string(index=False))

def save_table(data: pd.DataFrame, path: Path, **kwargs):
    if SAVE_OUTPUTS:
        data.to_csv(path, **kwargs)
    return path


def save_text(text: str, path: Path):
    if SAVE_OUTPUTS:
        path.write_text(text, encoding='utf-8')
    return path


def print_save_status(path: Path):
    if SAVE_OUTPUTS:
        print('-', path.relative_to(BASE_DIR))
    else:
        print(f'- {path.name}: 저장 비활성화, 노트북 출력만 확인')


## 4. Table 4A. 핵심 변수 상관표

`industry`는 명목형 변수이므로 제외한다. 본문용으로 Pearson 상관표를 저장하고, Spearman 상관표와 pairwise N도 별도 파일로 저장한다.

In [ ]:
corr_vars = [
    'effect_proc_improve', 'effect_average', 'it_org_any', 'ai_use',
    'ai_use_sum', 'it_invest_sum', 'firm_size'
]
num = df[corr_vars].apply(pd.to_numeric, errors='coerce')

pearson = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
spearman = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
pairwise_n = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=int)
pearson_p = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
spearman_p = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)

for a in corr_vars:
    for b in corr_vars:
        sub = num[[a]].dropna() if a == b else num[[a, b]].dropna()
        n = len(sub)
        pairwise_n.loc[a, b] = n
        if a == b:
            pearson.loc[a, b] = 1.0
            spearman.loc[a, b] = 1.0
            pearson_p.loc[a, b] = 0.0
            spearman_p.loc[a, b] = 0.0
        elif n >= 2:
            r = pearson_r(sub[a], sub[b])
            rs = pearson_r(sub[a].rank(method='average'), sub[b].rank(method='average'))
            pearson.loc[a, b] = r
            spearman.loc[a, b] = rs
            pearson_p.loc[a, b] = corr_p_value(r, n)
            spearman_p.loc[a, b] = corr_p_value(rs, n)

pearson_report = pd.DataFrame(index=corr_vars, columns=corr_vars)
for a in corr_vars:
    for b in corr_vars:
        pearson_report.loc[a, b] = report_corr_cell(pearson.loc[a, b], pearson_p.loc[a, b])
pearson_report_display = pearson_report.reset_index().rename(columns={'index': '변수명'})

save_table(pearson.round(3), OUT_DIR / 'table4a_correlation_pearson.csv', encoding='utf-8-sig')
save_table(spearman.round(3), OUT_DIR / 'table4a_correlation_spearman.csv', encoding='utf-8-sig')
save_table(pairwise_n, OUT_DIR / 'table4a_correlation_pairwise_n.csv', encoding='utf-8-sig')

show_table('Table 4A. Pearson correlation report table', pearson_report_display)
show_table('Table 4A pairwise N', pairwise_n.reset_index().rename(columns={'index': '변수명'}))

## 5. Table 4B. 가설별 집단 평균 비교표

H1, H2, AI 활용 여부 보조 확인을 위해 `it_org_any` 집단별 평균 또는 비율을 비교한다. 결측은 각 비교별 pairwise로 제외한다.

In [ ]:
def welch_t_p(group0, group1):
    x0 = pd.to_numeric(group0, errors='coerce').dropna().astype(float)
    x1 = pd.to_numeric(group1, errors='coerce').dropna().astype(float)
    n0, n1 = len(x0), len(x1)
    m0, m1 = x0.mean(), x1.mean()
    v0, v1 = x0.var(ddof=1), x1.var(ddof=1)
    se = math.sqrt(v0 / n0 + v1 / n1)
    z = (m1 - m0) / se if se else np.nan
    p = norm_two_sided_from_z(z)
    return m0, x0.std(ddof=1), n0, m1, x1.std(ddof=1), n1, m1 - m0, z, p


def prop_chi_p(group, outcome):
    sub = pd.DataFrame({'group': group, 'outcome': outcome}).dropna()
    sub['group'] = sub['group'].astype(int)
    sub['outcome'] = sub['outcome'].astype(int)
    tab = pd.crosstab(sub['group'], sub['outcome']).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    observed = tab.values.astype(float)
    expected = observed.sum(axis=1, keepdims=True) @ observed.sum(axis=0, keepdims=True) / observed.sum()
    chi2 = ((observed - expected) ** 2 / expected).sum()
    p = math.erfc(math.sqrt(chi2 / 2.0))
    p0 = tab.loc[0, 1] / tab.loc[0].sum()
    p1 = tab.loc[1, 1] / tab.loc[1].sum()
    return int(tab.loc[0].sum()), p0, int(tab.loc[1].sum()), p1, p1 - p0, chi2, p

rows = []
for comparison, outcome, metric in [
    ('H1 예비 확인', 'effect_proc_improve', '평균'),
    ('H2 예비 확인', 'ai_use_sum', '평균'),
]:
    sub = df[['it_org_any', outcome]].dropna()
    m0, sd0, n0, m1, sd1, n1, diff, stat, p = welch_t_p(
        sub.loc[sub['it_org_any'] == 0, outcome],
        sub.loc[sub['it_org_any'] == 1, outcome],
    )
    rows.append({
        '비교': comparison,
        '그룹 변수': 'it_org_any',
        '결과 변수': outcome,
        '지표': metric,
        '미보유(0) N': n0,
        '미보유(0) 평균/비율': m0,
        '미보유(0) SD': sd0,
        '보유(1) N': n1,
        '보유(1) 평균/비율': m1,
        '보유(1) SD': sd1,
        '차이(1-0)': diff,
        '검정': 'Welch t-test (정규근사 p)',
        '검정통계량': stat,
        'p-value': p,
        '유의표시': p_stars(p),
        '사용 N': len(sub),
    })

n0, p0, n1, p1, diff, chi2, p = prop_chi_p(df['it_org_any'], df['ai_use'])
rows.append({
    '비교': 'AI 활용 여부 보조 확인',
    '그룹 변수': 'it_org_any',
    '결과 변수': 'ai_use',
    '지표': '1의 비율',
    '미보유(0) N': n0,
    '미보유(0) 평균/비율': p0,
    '미보유(0) SD': np.nan,
    '보유(1) N': n1,
    '보유(1) 평균/비율': p1,
    '보유(1) SD': np.nan,
    '차이(1-0)': diff,
    '검정': 'Chi-square test (df=1)',
    '검정통계량': chi2,
    'p-value': p,
    '유의표시': p_stars(p),
    '사용 N': n0 + n1,
})

table4b = pd.DataFrame(rows)
for col in ['미보유(0) 평균/비율', '미보유(0) SD', '보유(1) 평균/비율', '보유(1) SD', '차이(1-0)', '검정통계량', 'p-value']:
    table4b[col] = table4b[col].astype(float).round(3)

table4b_display = table4b.copy()
table4b_display['p-value'] = table4b_display['p-value'].apply(lambda x: '<.001' if x < 0.001 else f'{x:.3f}')
save_table(table4b, OUT_DIR / 'table4b_group_mean_comparison.csv', index=False, encoding='utf-8-sig')

show_table('Table 4B. Group mean comparison', table4b_display)

## 6. Table 4C. 조절효과 예비 확인

`ai_use_sum_group`은 그림 및 예비 확인용 임시 변수이다. 원본 데이터셋에는 저장하지 않는다.

In [ ]:
interaction_df = df[['ai_use_sum', 'it_org_any', 'effect_proc_improve']].dropna().copy()
interaction_df['ai_use_sum_group'] = np.select(
    [interaction_df['ai_use_sum'].eq(0), interaction_df['ai_use_sum'].eq(1), interaction_df['ai_use_sum'].ge(2)],
    [0, 1, 2],
    default=np.nan,
).astype(int)

group_label = {0: '0개/미활용', 1: '1개', 2: '2개 이상'}

cell_rows = []
for (ai_group, org), sub in interaction_df.groupby(['ai_use_sum_group', 'it_org_any']):
    y = pd.to_numeric(sub['effect_proc_improve'], errors='coerce').dropna()
    n = len(y)
    mean = y.mean()
    sd = y.std(ddof=1)
    se = sd / math.sqrt(n) if n > 1 else np.nan
    ci = 1.96 * se if n > 1 else np.nan
    cell_rows.append({
        'ai_use_sum_group': ai_group,
        'AI 활용 수준': group_label[ai_group],
        'it_org_any': int(org),
        '정보화 담당 체계': '미보유' if int(org) == 0 else '보유',
        'N': n,
        '평균': mean,
        '표준편차': sd,
        '표준오차': se,
        '95% CI 하한': mean - ci,
        '95% CI 상한': mean + ci,
    })

table4c_cells = pd.DataFrame(cell_rows).sort_values(['ai_use_sum_group', 'it_org_any'])
for col in ['평균', '표준편차', '표준오차', '95% CI 하한', '95% CI 상한']:
    table4c_cells[col] = table4c_cells[col].round(3)

diff_rows = []
for ai_group, sub in interaction_df.groupby('ai_use_sum_group'):
    m0, sd0, n0, m1, sd1, n1, diff, stat, p = welch_t_p(
        sub.loc[sub['it_org_any'] == 0, 'effect_proc_improve'],
        sub.loc[sub['it_org_any'] == 1, 'effect_proc_improve'],
    )
    diff_rows.append({
        'ai_use_sum_group': ai_group,
        'AI 활용 수준': group_label[ai_group],
        '미보유(0) N': n0,
        '미보유(0) 평균': m0,
        '보유(1) N': n1,
        '보유(1) 평균': m1,
        '차이(1-0)': diff,
        't/z 통계량': stat,
        'p-value': p,
        '유의표시': p_stars(p),
        '사용 N': n0 + n1,
    })

table4c_diff = pd.DataFrame(diff_rows)
for col in ['미보유(0) 평균', '보유(1) 평균', '차이(1-0)', 't/z 통계량', 'p-value']:
    table4c_diff[col] = table4c_diff[col].astype(float).round(3)

table4c_diff_display = table4c_diff.copy()
table4c_diff_display['p-value'] = table4c_diff_display['p-value'].apply(lambda x: '<.001' if x < 0.001 else f'{x:.3f}')

save_table(table4c_cells, OUT_DIR / 'table4c_interaction_cell_means.csv', index=False, encoding='utf-8-sig')

show_table('Table 4C. Interaction cell means', table4c_cells)
show_table('Table 4C. Difference within AI use level', table4c_diff_display)

## 7. Markdown 보고서 및 해석 메모 저장

보고서에 붙여넣기 쉬운 Markdown 파일과 CSV 파일을 저장한다. 노트북 내부에도 주요 표와 해석 메모를 출력한다.

In [ ]:
# Table 4A Markdown report
pearson_md = [
    '# Table 4A. 핵심 변수 상관표 (Pearson)',
    '',
    f'- 사용 데이터: `{DATA_PATH.relative_to(BASE_DIR)}`',
    f'- 전체 N: {total_n:,}',
    '- 셀 값은 Pearson 상관계수와 유의수준 표시를 결합한 값이다. * p < .05, ** p < .01, *** p < .001.',
    '- p-value는 큰 표본에서 Fisher z 정규근사를 사용했다.',
    '',
    md_table(pearson_report_display),
    '',
    '## Pairwise N',
    md_table(pairwise_n.reset_index().rename(columns={'index': '변수명'})),
]
save_text('\n'.join(pearson_md) + '\n', OUT_DIR / 'table4a_correlation_report.md')

# Table 4B Markdown report
md4b = ['# Table 4B. 가설별 집단 평균 비교표', '', md_table(table4b_display)]
save_text('\n'.join(md4b) + '\n', OUT_DIR / 'table4b_group_mean_comparison_report.md')

# Table 4C Markdown report
md4c = [
    '# Table 4C. 조절효과 예비 확인',
    '',
    '## 셀 평균',
    md_table(table4c_cells),
    '',
    '## AI 활용 수준별 평균 차이',
    md_table(table4c_diff_display),
]
save_text('\n'.join(md4c) + '\n', OUT_DIR / 'table4c_interaction_report.md')

corr_it_effect = pearson.loc['it_org_any', 'effect_proc_improve']
corr_it_ai = pearson.loc['it_org_any', 'ai_use_sum']
corr_ai_effect = pearson.loc['ai_use_sum', 'effect_proc_improve']
h1 = table4b.loc[table4b['비교'] == 'H1 예비 확인'].iloc[0]
h2 = table4b.loc[table4b['비교'] == 'H2 예비 확인'].iloc[0]
ai_prop = table4b.loc[table4b['비교'] == 'AI 활용 여부 보조 확인'].iloc[0]
low_diff = table4c_diff.loc[table4c_diff['ai_use_sum_group'] == 0, '차이(1-0)'].iloc[0]
high_diff = table4c_diff.loc[table4c_diff['ai_use_sum_group'] == 2, '차이(1-0)'].iloc[0]

interpretation_sentences = [
    f'핵심 변수 간 Pearson 상관을 보면 it_org_any와 effect_proc_improve의 상관은 {corr_it_effect:.3f}, it_org_any와 ai_use_sum의 상관은 {corr_it_ai:.3f}, ai_use_sum과 effect_proc_improve의 상관은 {corr_ai_effect:.3f}으로 모두 양(+)의 방향을 보여 가설의 기본 방향성과 대체로 일치한다.',
    f'H1 예비 평균 비교에서 정보화 담당 체계 보유 기업의 프로세스 개선 효과 평균은 {h1["보유(1) 평균/비율"]:.3f}점, 미보유 기업은 {h1["미보유(0) 평균/비율"]:.3f}점으로 보유 기업이 {h1["차이(1-0)"]:.3f}점 높게 나타났다.',
    f'H2 예비 평균 비교에서 정보화 담당 체계 보유 기업의 AI 활용 유형 개수 평균은 {h2["보유(1) 평균/비율"]:.3f}개, 미보유 기업은 {h2["미보유(0) 평균/비율"]:.3f}개로 보유 기업이 {h2["차이(1-0)"]:.3f}개 높게 나타났다.',
    f'보조적으로 AI 활용 기업 비율도 보유 기업 {ai_prop["보유(1) 평균/비율"] * 100:.1f}%, 미보유 기업 {ai_prop["미보유(0) 평균/비율"] * 100:.1f}%로 보유 기업에서 더 높게 나타났다.',
    f'H3 예비 패턴에서는 정보화 담당 체계 보유-미보유의 프로세스 개선 효과 평균 차이가 AI 활용 0개 집단에서 {low_diff:.3f}점, AI 활용 2개 이상 집단에서 {high_diff:.3f}점으로 나타나 AI 활용 수준에 따라 차이가 커지는 양상이 관찰된다.',
    '다만 이는 단순 상관과 집단 평균 비교에 기반한 예비 분석이므로 인과적 해석이나 가설 지지 여부를 단정하기에는 제한이 있다.',
    '이후 분석에서는 기업 규모와 업종을 통제한 회귀분석을 통해 핵심 관계가 유지되는지 확인할 필요가 있다.',
]

interp_md = [
    '# 예비 연관성 분석 해석 메모',
    '',
    f'- 사용 데이터: `{DATA_PATH.relative_to(BASE_DIR)}`',
    f'- 전체 N: {total_n:,}',
    '',
]
interp_md.extend(f'- {sentence}' for sentence in interpretation_sentences)
save_text('\n'.join(interp_md) + '\n', OUT_DIR / 'preliminary_association_interpretation.md')

print('저장 완료')
for filename in [
    'table4a_correlation_pearson.csv',
    'table4a_correlation_spearman.csv',
    'table4a_correlation_pairwise_n.csv',
    'table4a_correlation_report.md',
    'table4b_group_mean_comparison.csv',
    'table4b_group_mean_comparison_report.md',
    'table4c_interaction_cell_means.csv',
    'table4c_interaction_report.md',
    'preliminary_association_interpretation.md',
]:
    print('-', (OUT_DIR / filename).relative_to(BASE_DIR))

## 8. 최종 출력

사용 데이터, 변수 확인 결과, 주요 표, 해석 메모, 확인 필요 사항을 노트북 내부에서 최종 확인한다.

In [ ]:
need_check = []
if missing_vars:
    need_check.append('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))
else:
    need_check.append('요청 변수 8개가 모두 존재함')

if int(variable_check['결측 N'].sum()) == 0:
    need_check.append('요청 변수 8개 모두 결측 N=0이며, 각 분석별 pairwise 제외로 인한 표본 손실 없음')
else:
    need_check.append('일부 변수에 결측이 있어 각 분석별 pairwise 제외 N을 확인해야 함')
need_check.append('p-value는 예비적 참고용이며, scipy 없이 큰 표본 정규근사로 계산함')

print('1. 사용한 데이터 파일명과 전체 N')
print('-', DATA_PATH.relative_to(BASE_DIR))
print('-', f'N = {total_n:,}')
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

print('\n2. 변수 존재 여부 체크 결과')
display(variable_check) if display else print(variable_check.to_string(index=False))

print('\n3. Table 4A Markdown 표')
print(md_table(pearson_report_display))

print('\n4. Table 4B Markdown 표')
print(md_table(table4b_display))

print('\n5. Table 4C Markdown 표')
print(md_table(table4c_diff_display))

print('\n6. 보고서용 해석 메모')
for sentence in interpretation_sentences:
    print('-', sentence)

print('\n7. 확인 필요 사항')
for note in need_check:
    print('-', note)